In [4]:
from openai import OpenAI
import os
import pandas as pd
import base64
import io
from PIL import Image
import json
import re

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model_name = 'qwen/qwen3-vl-30b'

prompt_method = "Zero-shot1-GEO"
metadata_df = pd.read_csv("merged_metadata.csv")

# Create fast lookup: merged_index -> row (assumes merged_index unique)
metadata_df["merged_index"] = metadata_df["merged_index"].astype(int)
meta_by_index = metadata_df.set_index("merged_index", drop=False)

image_dir = "streetview"

character_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "Judgment",
        "schema": {
            "type": "object",
            "properties": {
                "Judgment": {"type": "string"},
                "Reasons": {"type": "string"}
            },
            "required": ["Judgment", "Reasons"]
        },
    }
}

results = []

# List images that actually exist
image_files = sorted(
    f for f in os.listdir(image_dir)
    if f.startswith("merged_") and f.lower().endswith(".jpg")
)

pattern = re.compile(r"merged_(\d{4})\.jpg", re.IGNORECASE)

for fname in image_files:
    m = pattern.match(fname)
    if not m:
        continue

    merged_index = int(m.group(1))
    if merged_index not in meta_by_index.index:
        # Image exists but no metadata row; skip
        continue

    row = meta_by_index.loc[merged_index]

    study_question = row["study_question"]
    left_place = row["place_name_left"]
    right_place = row["place_name_right"]
    ground_truth = str(row["choice"]).strip().lower()
    image_path = os.path.join(image_dir, fname)

    # Encode image to base64
    image = Image.open(image_path).convert("RGB")
    img_byte_arr = io.BytesIO()
    image.save(img_byte_arr, format="JPEG")
    base64_image = base64.b64encode(img_byte_arr.getvalue()).decode("utf-8")

    prompt_text = f"""
You are shown a side-by-side image with two street views at two different cities: the left half at {left_place} and the right half at {right_place}.
Which side looks more {study_question}?

Answer with only one word: left or right. Then explain your reasoning.

Return JSON with keys: Judgment, Reasons.
""".strip()

    response = client.chat.completions.create(
        model=model_name,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ],
        }],
        response_format=character_schema,
    )

    raw = response.choices[0].message.content.strip()
    raw = (raw.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'"))

    # Parse JSON safely
    model_judgement, model_reason = "", ""
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            model_judgement = parsed.get("Judgment", "").strip().lower()
            model_reason = parsed.get("Reasons", "").strip()
        except json.JSONDecodeError:
            pass

    print(merged_index, model_judgement, model_reason)

    results.append({
        "merged_index": merged_index,
        "left": row["left"],
        "right": row["right"],
        "study_question": study_question,
        "ground_truth": ground_truth,
        "left_vote": row["left_vote"],
        "right_vote": row["right_vote"],
        "model_judgement": model_judgement,
        "model_reason": model_reason,
        "validation": int(model_judgement == ground_truth)
    })

df_result = pd.DataFrame(results)
df_result.to_csv(f"llm_predictions.csv", index=False)

accuracy_all = df_result["validation"].mean() if len(df_result) else float("nan")
filtered_df = df_result[(df_result["ground_truth"] != "equal") & (df_result["model_judgement"] != "equal")]
accuracy_excl_equal = filtered_df["validation"].mean() if len(filtered_df) else float("nan")

print(f"✅ Processed images: {len(df_result)}")
print(f"✅ Accuracy (all): {accuracy_all:.2%}")
print(f"✅ Accuracy (excluding 'equal'): {accuracy_excl_equal:.2%}")

0 left The left side, depicting Paris, appears more lively due to the presence of a wide, tree-lined street with a large grassy area, modern buildings, and visible pedestrian activity. In contrast, the right side, showing Dublin, features a narrower, less developed street with fewer people and vehicles, giving it a quieter, less vibrant impression.
1 left The left side (Boston) appears safer due to several factors: the road is wider and clearer with no parked cars obstructing visibility, there are no large trucks or construction vehicles present, the sidewalks appear more maintained, and the overall environment looks less congested and more orderly. In contrast, the right side (Sydney) shows a busy street with multiple parked cars, a large cement mixer truck, and a higher density of vehicles, which can increase the risk of accidents and reduce driver visibility. Additionally, the Boston side has fewer potential hazards and a more controlled traffic environment.
✅ Processed images: 2
✅ 